# The LLM Model Executor

By the end of this notebook, you will understand the role of the `Executor` in a large language model serving system. You will see how it acts as a clean abstraction layer between the high-level scheduling logic and the low-level model execution on hardware. We will build a simplified `Executor` from scratch to see how it takes a batch of sequences, simulates a forward pass, and returns the results.

In [ ]:
# === Setup Cell ===
# All necessary imports for the notebook.

import torch
import dataclasses
from typing import List, Dict, Tuple

# For reproducibility
torch.manual_seed(42)

# A simple dataclass to represent a single sequence in a batch.
# In a real system, this would be more complex, holding KV cache info.
@dataclasses.dataclass
class Sequence:
    seq_id: int
    token_ids: List[int]

# A dataclass for the batch the Scheduler sends to the Executor.
@dataclasses.dataclass
class ScheduledBatch:
    sequences: List[Sequence]
    # In the real system, this would contain detailed KV cache block mappings.
    # Here, we simplify it to just the sequences.

# The output from the Executor for each sequence.
@dataclasses.dataclass
class SequenceOutput:
    seq_id: int
    # The logits for the *next* token prediction.
    next_token_logits: torch.Tensor

### The Core Idea: The Specialist Chef

Imagine a high-tech, incredibly busy restaurant kitchen. The `Scheduler` is the head chef, constantly juggling incoming orders (requests), figuring out which orders can be cooked together to save time, and preparing trays of ingredients (token data and KV cache pointers).

But the head chef doesn't personally operate the restaurant's most advanced and complicated piece of equipment: a massive, fusion-powered molecular gastronomy oven (the GPU). For that, there's a specialist: the **Executor**.

The `Executor`'s job is singular and focused. The head chef (`Scheduler`) brings it a fully prepared tray (`ScheduledBatch`). The `Executor` takes this tray, operates the complex oven (`model.forward()`), and cooks everything perfectly. It then returns the tray of cooked dishes (`SequenceOutput` with logits) back to the head chef.

This division of labor is crucial:

1.  **Abstraction:** The head chef doesn't need to know the oven's intricate settings or maintenance schedule. It just needs to know how to hand off a tray and get food back.
2.  **Specialization:** The `Executor` is an expert at one thing: running the model efficiently on the hardware. It doesn't get distracted by customer wait times or new orders arriving.

In vLLM, the `Executor` is this specialist. It takes a batch of sequences prepared by the `Scheduler`, runs the model's forward pass on the GPU, and returns the resulting logits, hiding all the complexity of the actual computation.

In [ ]:
# A mock LLM that simulates a real model's forward pass.
class MockLLM:
    """
    A stand-in for a real large language model like Llama or GPT.
    Its job is to take token IDs and produce logits.
    """
    def __init__(self, vocab_size: int = 100):
        self.vocab_size = vocab_size
        print(f"MockLLM: Model 'loaded' (vocab size: {self.vocab_size}).")

    def forward(self, batch_token_ids: torch.Tensor) -> torch.Tensor:
        """
        Simulates a batched forward pass.
        In a real model, this would be a complex series of matrix multiplications.
        Here, we just generate random data with the correct shape.
        """
        batch_size, num_tokens = batch_token_ids.shape
        print(f"MockLLM: Running forward pass on a batch of size {batch_size}...")
        # We return logits only for the *last* token in each sequence.
        # Shape: [batch_size, vocab_size]
        mock_logits = torch.randn(batch_size, self.vocab_size)
        return mock_logits


class Executor:
    """
    The Executor manages the model and runs the forward pass.
    It abstracts away the details of the actual computation.
    """
    def __init__(self, model: MockLLM):
        # The executor "owns" the model and the "device" it runs on.
        self.model = model

    def execute_model(self, batch: ScheduledBatch) -> List[SequenceOutput]:
        """
        The main entry point for running a computation.
        It takes a batch from the scheduler and returns the results.
        """
        # 1. Prepare inputs for the model (e.g., convert to a tensor).
        # We assume all sequences in the batch are for a single step,
        # so we only need the last token ID from each.
        # In a real system, you'd pad and stack them.
        token_ids_list = [seq.token_ids for seq in batch.sequences]
        input_tensor = torch.tensor(token_ids_list, dtype=torch.long)

        # 2. Run the model's forward pass.
        logits = self.model.forward(input_tensor) # [batch_size, vocab_size]

        # 3. Process the output, mapping results back to sequence IDs.
        outputs = []
        for i, seq in enumerate(batch.sequences):
            seq_logits = logits[i]
            outputs.append(SequenceOutput(seq_id=seq.seq_id, next_token_logits=seq_logits))

        return outputs

### Walking Through the Implementation

Let's break down our simple `Executor` and its `MockLLM`.

**`MockLLM`**

-   `__init__(self, vocab_size)`: We initialize our fake model with a `vocab_size`. This determines the shape of the output logits tensor, which is crucial for the rest of the system.
-   `forward(self, batch_token_ids)`: This is the heart of the simulation. It takes a PyTorch tensor of token IDs, where each row is a different sequence in the batch. It prints a message to show it's running and then returns a tensor of random "logits". The output shape `[batch_size, vocab_size]` is exactly what a real transformer would produce after a forward pass when predicting the next token for each sequence.

**`Executor`**

-   `__init__(self, model)`: The `Executor` is created with a reference to the model it's supposed to run. This encapsulates the model, meaning other parts of the system don't touch the model directly; they go through the `Executor`.
-   `execute_model(self, batch)`: This is the `Executor`'s primary job.
    1.  **Prepare Inputs**: It receives a `ScheduledBatch` object, which is just a list of `Sequence` objects. It extracts the `token_ids` from each sequence and bundles them into a single tensor, `input_tensor`. This step is called "batching".
    2.  **Run Forward Pass**: It calls `self.model.forward()` with the prepared batch tensor. This is the moment the "computation" happens. The `Executor` waits for the model to finish and receives the `logits` tensor.
    3.  **Process Outputs**: The model returns a single, batched tensor of logits. The `Executor`'s final task is to "un-batch" the results. It iterates through the original sequences and the output logits, pairing them up and wrapping them in `SequenceOutput` objects. This makes it easy for the next stage (the `OutputProcessor`) to know which result belongs to which request.

This clean separation of concerns is key. The `Scheduler` worries about *what* to run, and the `Executor` worries about *how* to run it.

In [ ]:
# === Demonstration ===

# 1. Set up the components.
print("--- Initializing ---")
mock_model = MockLLM(vocab_size=50)
executor = Executor(model=mock_model)
print("-" * 20)


# 2. Create a batch of work, as if it came from the Scheduler.
# Note the different sequence lengths, which is common.
print("--- Creating a batch ---")
batch_to_run = ScheduledBatch(sequences=[
    Sequence(seq_id=101, token_ids=[1, 2, 3]),
    Sequence(seq_id=256, token_ids=[8, 9, 10, 11]),
    Sequence(seq_id=314, token_ids=[5, 6]),
])
print(f"Batch contains {len(batch_to_run.sequences)} sequences.")
print("-" * 20)


# 3. Ask the Executor to run the model on the batch.
print("--- Executing the batch ---")
outputs = executor.execute_model(batch_to_run)
print("-" * 20)


# 4. Inspect the results.
print("--- Inspecting the outputs ---")
for output in outputs:
    print(f"Sequence ID: {output.seq_id}")
    # The shape should be [vocab_size], which is 50 in our case.
    print(f"  Logits shape: {output.next_token_logits.shape}")
    # Let's see a few example logit values.
    print(f"  Example logits: {output.next_token_logits[:5].tolist()}")
    print()

# We can see that the Executor correctly processed the batch and returned
# a separate output object for each sequence that was in the input batch.

### Going Deeper: Why a Separate Executor?

A fair question is: why have an `Executor` class at all? Couldn't the `Scheduler` just call `model.forward()` directly?

The answer lies in building a system that is flexible and can evolve. The `Executor` acts as a crucial **abstraction barrier**. The `Scheduler`'s job is complex enough; it shouldn't also need to know the gritty details of *how* the model is physically run.

Consider these scenarios:

1.  **Single GPU (our current model):** The `Executor` just calls `model.forward()`. Simple.
2.  **Multi-GPU with Tensor Parallelism:** A single giant model is split across 4 GPUs. To run a forward pass, the `Executor` must manage complex communication (e.g., `all_reduce` operations) between the GPUs. The `Scheduler` has no idea this is happening; it just calls `executor.execute_model()` as always.
3.  **Multiple Model Replicas:** You have multiple copies of the same model on different GPUs to serve more traffic. The `Executor` might contain logic to dispatch the batch to the next available GPU replica.

By hiding this complexity behind a consistent `execute_model` interface, we can change the entire backend execution strategy without touching the scheduling logic. This is a powerful design principle that keeps the system clean and maintainable.

Let's illustrate this with placeholder code.

In [ ]:
# A placeholder for a more complex executor.
class MultiGPUExecutor:
    """
    A hypothetical Executor for a model split across multiple GPUs.
    Its public interface is IDENTICAL to the simple Executor.
    """
    def __init__(self, model_on_gpu1, model_on_gpu2):
        # In reality, these would be model shards (e.g., layers)
        self.shard1 = model_on_gpu1
        self.shard2 = model_on_gpu2
        print("MultiGPUExecutor: Ready with 2 model shards.")

    def execute_model(self, batch: ScheduledBatch) -> List[SequenceOutput]:
        """
        The interface is the same, but the internal logic is far more complex.
        """
        print("MultiGPUExecutor: Executing across multiple devices...")
        token_ids_list = [seq.token_ids for seq in batch.sequences]
        input_tensor = torch.tensor(token_ids_list, dtype=torch.long)

        # ---- FAKE MULTI-GPU LOGIC ----
        # 1. Run first half of the model on "GPU 1"
        intermediate_activations = self.shard1.forward(input_tensor)
        # 2. Communicate/transfer activations from GPU 1 to GPU 2 (not shown)
        print("   (Simulating communication between GPUs...)")
        # 3. Run second half of the model on "GPU 2"
        logits = self.shard2.forward(intermediate_activations)
        # -----------------------------

        outputs = []
        for i, seq in enumerate(batch.sequences):
            outputs.append(SequenceOutput(seq_id=seq.seq_id, next_token_logits=logits[i]))

        return outputs


# The scheduler's code doesn't need to change at all!
# It can use the simple or complex executor interchangeably.

print("--- Using the simple Executor ---")
simple_executor = Executor(model=MockLLM(vocab_size=10))
simple_executor.execute_model(batch_to_run)

print("\n" + "="*40 + "\n")

print("--- Using the Multi-GPU Executor ---")
# Let's pretend two MockLLMs are two shards of a bigger model
shard1 = MockLLM(vocab_size=10)
shard2 = MockLLM(vocab_size=10)
multi_gpu_executor = MultiGPUExecutor(shard1, shard2)
multi_gpu_executor.execute_model(batch_to_run)

### Visualization: A Batch's Journey

Let's visualize the flow of data for a single call to `execute_model`. The `Scheduler` assembles a batch of unrelated sequences. The `Executor` treats them as one big computational job, runs the model, and then maps the results back to their original sequence IDs.

This ASCII art shows a batch of three sequences entering the `Executor`, being processed by the model in a single forward pass, and emerging as individual outputs.

In [ ]:
import random

# --- ASCII Art Visualization ---

def visualize_execution(batch: ScheduledBatch, outputs: List[SequenceOutput]):
    vis = []
    vis.append("Scheduler Creates Batch:")
    for seq in batch.sequences:
        vis.append(f"  - Seq {seq.seq_id}: {seq.token_ids}")
    vis.append("     |")
    vis.append("     v")
    vis.append(" [ Executor ]------------------------------------")
    vis.append(" |                                              |")
    vis.append(" |  1. Collects token IDs into a single tensor    |")
    vis.append(" |     [1, 2, 3]                                  |")
    vis.append(" |     [8, 9, 10, 11] -> torch.tensor(...)        |")
    vis.append(" |     [5, 6]                                     |")
    vis.append(" |                                              |")
    vis.append(" |  2. Calls model.forward(tensor) on GPU         |")
    vis.append(" |     ...Computation happening...                |")
    vis.append(" |     Logits Tensor Returned [3, vocab_size]     |")
    vis.append(" |                                              |")
    vis.append(" |  3. Maps results back to sequence IDs        |")
    vis.append(" |                                              |")
    vis.append(" ------------------------------------------------")
    vis.append("     |")
    vis.append("     v")
    vis.append("Executor Returns Individual Outputs:")
    for out in outputs:
        # Let's find the predicted token for a more interesting output
        predicted_token_id = torch.argmax(out.next_token_logits).item()
        vis.append(f"  - For Seq {out.seq_id}: Predicted Token ID = {predicted_token_id}")

    print("\n".join(vis))

# We can reuse the batch and outputs from our earlier demonstration.
print("Visualizing the journey of our previously executed batch:")
visualize_execution(batch_to_run, outputs)

### Summary

In this notebook, we've explored the role of the `Executor`. It's the component that acts as the bridge between the high-level logic of the serving system and the low-level execution of the model on the hardware.

**What We Built:**
-   A `MockLLM` to simulate the behavior of a real transformer model.
-   A simple `Executor` that takes a `ScheduledBatch`, runs the model, and returns a list of `SequenceOutput`.
-   A conceptual `MultiGPUExecutor` to demonstrate how the `Executor` interface provides a powerful abstraction.

**Key Takeaways:**
-   The `Executor`'s primary responsibility is to execute a model's forward pass.
-   It **abstracts away** the complexity of the underlying hardware and model architecture (e.g., single vs. multi-GPU).
-   Its clean interface (`execute_model`) decouples the "what to run" (Scheduler) from the "how to run it" (Executor), making the entire system more modular and maintainable.

**What's Next:**
Now that we have a component that can execute a batch of work, where do these batches come from? In the next notebook, **Request Scheduling and Continuous Batching**, we'll build the `Scheduler`, the brains of the operation that decides which requests to group together to send to the `Executor`.